In [1]:
import joblib


In [ ]:
from flask import Flask, request, jsonify
import joblib
import pandas as pd
from math import radians, sin, cos, sqrt, atan2
import numpy as np
import traceback
import os
app = Flask(__name__)
from flask_cors import CORS

CORS(app)

# Load the trained model
# Make sure 'xgboost_model.joblib' is in the same directory or provide the full path
MODEL_DIR = os.path.dirname(os.path.abspath(__file__)) if '__file__' in dir() else os.getcwd()
model_path = os.path.join(MODEL_DIR, 'xgboost_model.joblib')
model = joblib.load(model_path)
print(f'[price] Model loaded from: {model_path}')
try:
    print(f'[price] Model feature names: {model.get_booster().feature_names}')
except Exception:
    print('[price] Could not read feature names from model')

@app.route('/price', methods=['POST'])
def price():
    try:
        # Get data from POST request
        data = request.get_json(force=True)
        
        # Convert dictionary to DataFrame
        # Ensure the column order matches the training data
        # You might need to adjust this based on your X_train column names
        # Example: {'feature1': value1, 'feature2': value2, ...}
        # For simplicity, assuming data is a list of dictionaries for multiple predictions
        # or a single dictionary for one prediction.
        if isinstance(data, dict):
            input_df = pd.DataFrame([data])
        elif isinstance(data, list):
            input_df = pd.DataFrame(data)
        else:
            return jsonify({'error': 'Invalid input format. Expected dictionary or list of dictionaries.'}), 400

        print(f'[price] Received columns: {list(input_df.columns)}')

        try:
            expected = model.get_booster().feature_names
            missing = [f for f in expected if f not in input_df.columns]
            extra = [f for f in input_df.columns if f not in expected]
            if missing:
                print(f'[price] Missing features: {missing}')
            if extra:
                print(f'[price] Extra features (ignored): {extra}')
            if missing:
                return jsonify({'error': 'Missing features', 'missing': missing, 'expected': expected}), 400
        except Exception:
            pass

        # Make predictions
        predictions = model.predict(input_df)
        
        # Convert predictions to a list or array for JSON serialization
        return jsonify(predictions.tolist())

    except Exception as e:
        print(f'[price] ERROR: {e}')
        traceback.print_exc()
        return jsonify({'error': str(e)}), 500

saved = joblib.load("xgboost_tip_model.pkl")
model2 = saved["model"]
feature_columns = saved["feature_columns"]


@app.route("/tips", methods=["POST"])
def tips():
    try:
        data = request.get_json()

        if isinstance(data, dict):
            data = [data]

        df = pd.DataFrame(data)

        missing = [col for col in feature_columns if col not in df.columns]
        extra = [col for col in df.columns if col not in feature_columns]

        if missing:
            return jsonify({
                "error": "Missing features",
                "missing": missing
            }), 400

        df = df[feature_columns]

        predictions = model2.predict(df)

        return jsonify({
            "predictions": predictions.tolist()
        })

    except Exception as e:
        print(f'[tips] ERROR: {e}')
        traceback.print_exc()
        return jsonify({"error": str(e)}), 500

model3 = joblib.load("nyc_taxi_model.pkl")

def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = sin(dlat/2)**2 + cos(lat1)*cos(lat2)*sin(dlon/2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1-a))

def calculate_bearing(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlon = lon2 - lon1
    x = sin(dlon) * cos(lat2)
    y = cos(lat1)*sin(lat2) - sin(lat1)*cos(lat2)*cos(dlon)
    return np.degrees(np.arctan2(x, y)) % 360

@app.route('/duration', methods=['POST'])
def duration():
    try:
        data = request.get_json(force=True)
        if isinstance(data, list):
            trip = data[0]
        else:
            trip = data

        distance_km = haversine(
            trip['pickup_latitude'], trip['pickup_longitude'],
            trip['dropoff_latitude'], trip['dropoff_longitude']
        )

        trip_hours = trip['trip_duration_seconds'] / 3600
        speed_haversine = distance_km / trip_hours if trip_hours > 0 else 0

        bearing   = calculate_bearing(
            trip['pickup_latitude'], trip['pickup_longitude'],
            trip['dropoff_latitude'], trip['dropoff_longitude']
        )
        center_lat = (trip['pickup_latitude']  + trip['dropoff_latitude'])  / 2
        center_lon = (trip['pickup_longitude'] + trip['dropoff_longitude']) / 2

        h = trip['pickup_hour']
        m = trip['pickup_month']
        d = trip['pickup_dayofweek']

        hour_sin  = np.sin(2 * np.pi * h / 24)
        hour_cos  = np.cos(2 * np.pi * h / 24)
        month_sin = np.sin(2 * np.pi * m / 12)
        month_cos = np.cos(2 * np.pi * m / 12)
        dow_sin   = np.sin(2 * np.pi * d / 7)
        dow_cos   = np.cos(2 * np.pi * d / 7)

        is_rush_hour = int(h in [7, 8, 9, 17, 18, 19])
        is_night     = int(h in [0, 1, 2, 3, 4, 5])

        df = pd.DataFrame([{
            "vendor_id"           : trip['vendor_id'],
            "passenger_count"     : trip['passenger_count'],
            "pickup_longitude"    : trip['pickup_longitude'],
            "pickup_latitude"     : trip['pickup_latitude'],
            "dropoff_longitude"   : trip['dropoff_longitude'],
            "dropoff_latitude"    : trip['dropoff_latitude'],
            "store_and_fwd_flag"  : trip['store_and_fwd_flag'],
            "pickup_month"        : m,
            "pickup_day"          : trip['pickup_day'],
            "pickup_dayofweek"    : d,
            "pickup_hour"         : h,
            "pickup_minute"       : trip['pickup_minute'],
            "pickup_second"       : trip['pickup_second'],
            "is_weekend"          : trip['is_weekend'],
            "bearing"             : bearing,
            "center_lat"          : center_lat,
            "center_lon"          : center_lon,
            "is_rush_hour"        : is_rush_hour,
            "is_night"            : is_night,
            "hour_sin"            : hour_sin,
            "hour_cos"            : hour_cos,
            "month_sin"           : month_sin,
            "month_cos"           : month_cos,
            "dow_sin"             : dow_sin,
            "dow_cos"             : dow_cos,
            "speed_haversine"     : speed_haversine,
        }])

        prediction_log = model3.predict(df)[0]
        prediction_sec = int(np.expm1(prediction_log))
        prediction_min = round(prediction_sec / 60, 2)

        return jsonify({
            "trip_duration_seconds": prediction_sec,
            "trip_duration_minutes": prediction_min
        })

    except Exception as e:
        print(f'[duration] ERROR: {e}')
        traceback.print_exc()
        return jsonify({'error': str(e)}), 500

# =========================
# BUSY MODEL
# =========================

saved_busy = joblib.load("rf_model_bundle.pkl")

busy_model = saved_busy["model"]
area_encoder = saved_busy["le_area"]
label_encoder = saved_busy["le"]


@app.route('/busy', methods=['POST'])
def busy():
    try:
        data = request.get_json(force=True)

        lat     = data["lat"]
        lon     = data["lon"]
        hour    = data["hour"]
        day     = data["day"]
        month   = data["month"]
        weekday = data["weekday"]

        grid_size = 0.04
        lat_grid  = int(lat / grid_size)
        lon_grid  = int(lon / grid_size)
        area      = f"{lat_grid}_{lon_grid}"

        area_encoded = area_encoder.transform([area])[0] if area in area_encoder.classes_ else -1

        X = pd.DataFrame([{
            "Lat"          : lat,
            "Lon"          : lon,
            "hour"         : hour,
            "day"          : day,
            "month"        : month,
            "weekday"      : weekday,
            "area_encoded" : area_encoded
        }])

        prediction_encoded = busy_model.predict(X)[0]
        prediction_label   = label_encoder.inverse_transform([prediction_encoded])[0]

        return jsonify({
            "lat"        : lat,
            "lon"        : lon,
            "hour"       : hour,
            "area"       : area,
            "busy_class" : prediction_label
        })

    except Exception as e:
        print(f'[busy] ERROR: {e}')
        traceback.print_exc()
        return jsonify({"error": str(e)}), 500

@app.route('/busy/batch', methods=['POST'])
def busy_batch():
    try:
        data = request.get_json(force=True)
        points = data["points"]   # list of {lat, lon}
        hour    = data["hour"]
        day     = data["day"]
        month   = data["month"]
        weekday = data["weekday"]

        rows = []
        for p in points:
            lat, lon = p["lat"], p["lon"]
            grid_size = 0.04
            lat_grid = int(lat / grid_size)
            lon_grid = int(lon / grid_size)
            area = f"{lat_grid}_{lon_grid}"
            area_encoded = area_encoder.transform([area])[0] if area in area_encoder.classes_ else -1
            rows.append({
                "Lat": lat, "Lon": lon,
                "hour": hour, "day": day,
                "month": month, "weekday": weekday,
                "area_encoded": area_encoded
            })

        df = pd.DataFrame(rows)
        preds = busy_model.predict(df)
        labels = label_encoder.inverse_transform(preds)

        return jsonify([
            {"lat": points[i]["lat"], "lon": points[i]["lon"], "busy_class": labels[i]}
            for i in range(len(points))
        ])

    except Exception as e:
        traceback.print_exc()
        return jsonify({"error": str(e)}), 500

if __name__ == '__main__':
    app.run()

[price] Model loaded from: c:\jp\jp\src\MODEL_API\xgboost_model.joblib
[price] Model feature names: ['pickup_longitude', 'pickup_latitude', 'dropoff_longitude', 'dropoff_latitude', 'pickup_hour', 'pickup_day', 'pickup_month', 'pickup_dayofweek', 'distance']
 * Serving Flask app '__main__'
 * Debug mode: off


c:\Users\hezmi\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeClassifier from version 1.8.0 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\hezmi\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator RandomForestClassifier from version 1.8.0 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\hezmi\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to 

[price] Received columns: ['pickup_longitude', 'pickup_latitude', 'dropoff_longitude', 'dropoff_latitude', 'pickup_hour', 'pickup_day', 'pickup_month', 'pickup_dayofweek', 'distance']


127.0.0.1 - - [02/Jun/2026 00:36:53] "POST /price HTTP/1.1" 200 -
127.0.0.1 - - [02/Jun/2026 00:36:53] "POST /duration HTTP/1.1" 200 -


[price] Received columns: ['pickup_longitude', 'pickup_latitude', 'dropoff_longitude', 'dropoff_latitude', 'pickup_hour', 'pickup_day', 'pickup_month', 'pickup_dayofweek', 'distance']


...

127.0.0.1 - - [02/Jun/2026 00:58:12] "POST /duration HTTP/1.1" 200 -
127.0.0.1 - - [02/Jun/2026 00:58:12] "POST /price HTTP/1.1" 200 -


[price] Received columns: ['pickup_longitude', 'pickup_latitude', 'dropoff_longitude', 'dropoff_latitude', 'pickup_hour', 'pickup_day', 'pickup_month', 'pickup_dayofweek', 'distance']


127.0.0.1 - - [02/Jun/2026 00:58:14] "POST /price HTTP/1.1" 200 -
127.0.0.1 - - [02/Jun/2026 00:58:14] "POST /duration HTTP/1.1" 200 -


[price] Received columns: ['pickup_longitude', 'pickup_latitude', 'dropoff_longitude', 'dropoff_latitude', 'pickup_hour', 'pickup_day', 'pickup_month', 'pickup_dayofweek', 'distance']


127.0.0.1 - - [02/Jun/2026 00:58:32] "POST /price HTTP/1.1" 200 -
127.0.0.1 - - [02/Jun/2026 00:58:32] "POST /duration HTTP/1.1" 200 -
127.0.0.1 - - [02/Jun/2026 00:59:08] "POST /tips HTTP/1.1" 400 -
